In [1]:
# multi_fasta_dataset_biopython.py

import torch
from torch.utils.data import Dataset
from Bio import SeqIO 
import pandas as pd


def get_peptide_window(sequence, position, upstream=6, downstream=4, pad_str='X'):
    """
    Returns a window of amino acids around a specified position, handling padding at the sequence ends.

    The residue at the given 'position' is considered the cleavage site and is the
    last residue in the upstream part of the window.

    Args:
        sequence (str): The full protein sequence.
        position (int): The 0-based index of the cleavage site residue.
        upstream (int): Number of residues to include in the upstream segment (including the site).
        downstream (int): Number of residues to include in the downstream segment.
        pad_str (str): The character to use for padding.

    Returns:
        str: The resulting peptide window.
    """
    seq_len = len(sequence)

    if not (0 <= position < seq_len):
        return None
    # The start of the window is 'upstream' residues before the position
    start_pos = position - upstream + 1
    # The end of the window is 'downstream' residues after the position
    end_pos = position + 1 + downstream


    # Take the slice from the sequence
    # This handles cases where start_pos is negative or end_pos is beyond seq_len
    window_slice = sequence[max(0, start_pos) : min(seq_len, end_pos)]

    # Calculate how much padding is needed on each side
    pad_left = max(0, -start_pos)
    pad_right = max(0, end_pos - seq_len)

    # Construct the final padded window
    final_window = (pad_str * pad_left) + window_slice + (pad_str * pad_right)

    return final_window


class MultiFastaWindowDataset(Dataset):
    def __init__(self, fasta_file, upstream=6, downstream=4):
        super().__init__()
        self.upstream = upstream
        self.downstream = downstream
        
        # Parse tfasta file into a dictionary in memory {ID:SeqRecord}
        print("Loading FASTA file into memory with BioPython...")
        self.records = SeqIO.to_dict(SeqIO.parse(fasta_file, "fasta"))
        
        # iterates over the dictionary's values
        self.index = []
        for record in self.records.values():
            # We iterate up to the last possible cleavage site
            for i in range(len(record.seq) ):
                self.index.append((record.id, i))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        prot_name, position = self.index[idx]
        
        # acces in-memory dictionary of biopython
        sequence = str(self.records[prot_name].seq)
        
        # This part remains the same
        window = get_peptide_window(
            sequence,
            position=position,
            upstream=self.upstream,
            downstream=self.downstream
        )
        
        return {
            "prot_id": prot_name,
            "position": position + 1,
            "window": window
        }

In [2]:
UPSTREAM = 6
DOWNSTREAM = 4
dataset = MultiFastaWindowDataset(
    fasta_file="vax_seqs_test.fasta",
    upstream=UPSTREAM,
    downstream=DOWNSTREAM
)

for i in range(30):
    print(dataset[i])

Loading FASTA file into memory with BioPython...
{'prot_id': 'seq0_EAAY', 'position': 1, 'window': 'XXXXXCALNN'}
{'prot_id': 'seq0_EAAY', 'position': 2, 'window': 'XXXXCALNNE'}
{'prot_id': 'seq0_EAAY', 'position': 3, 'window': 'XXXCALNNEA'}
{'prot_id': 'seq0_EAAY', 'position': 4, 'window': 'XXCALNNEAA'}
{'prot_id': 'seq0_EAAY', 'position': 5, 'window': 'XCALNNEAAY'}
{'prot_id': 'seq0_EAAY', 'position': 6, 'window': 'CALNNEAAYD'}
{'prot_id': 'seq0_EAAY', 'position': 7, 'window': 'ALNNEAAYDP'}
{'prot_id': 'seq0_EAAY', 'position': 8, 'window': 'LNNEAAYDPL'}
{'prot_id': 'seq0_EAAY', 'position': 9, 'window': 'NNEAAYDPLV'}
{'prot_id': 'seq0_EAAY', 'position': 10, 'window': 'NEAAYDPLVV'}
{'prot_id': 'seq0_EAAY', 'position': 11, 'window': 'EAAYDPLVVA'}
{'prot_id': 'seq0_EAAY', 'position': 12, 'window': 'AAYDPLVVAA'}
{'prot_id': 'seq0_EAAY', 'position': 13, 'window': 'AYDPLVVAAS'}
{'prot_id': 'seq0_EAAY', 'position': 14, 'window': 'YDPLVVAASI'}
{'prot_id': 'seq0_EAAY', 'position': 15, 'window':

In [12]:
18000*60

1080000